In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    """Extracts text from a given PDF file."""
    doc = fitz.open(pdf_path)  # Open the PDF file
    text = "\n".join([page.get_text("text") for page in doc])  # Extract text from each page
    return text

# 🔹 Provide the PDF file path here
pdf_path = "Meta_Report2024.pdf"  # Change this to your actual file path
pdf_text = extract_text_from_pdf(pdf_path)

# 🔹 Preview first 1000 characters to check extraction
print(pdf_text[:1000])


In [ ]:
import os

os.environ["GROQ_API_KEY"] = input("Enter your GROQ API Key: ")

In [ ]:
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent
from langchain.tools import Tool
from langchain.schema import AIMessage, HumanMessage

In [ ]:
# Initialize Chat Model
chat_model = ChatGroq(temperature=0, model_name="llama3-70b-8192")

# Memory for Context Retention
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
# Initialize Chat History
memory.chat_memory.add_user_message("Start analyzing financial data")
memory.chat_memory.add_ai_message("I am ready to analyze. Please provide the data.")

# Define PDF Processing Tool
def analyze_financial_report():
    """Analyzes financial data from a report."""
    prompt = f"""
    You are a financial analyst. Analyze the following financial report text and provide insights:
    {pdf_text[:4000]}  #Limit input to 4000 characters to fit model constraints
    """
    response = chat_model.invoke(prompt)
    return response.content

pdf_tool = Tool(
    name="Financial Report Analysis",
    func=lambda _: analyze_financial_report(),
    description="Extracts and analyzes key financial data from reports."
)

In [ ]:
#Initialize Agent
agent = initialize_agent(
    agent="chat-conversational-react-description",
    tools=[pdf_tool],
    llm=chat_model,
    memory=memory,
    verbose=True
)

#Run Query on the PDF Data
response = agent.run("Summarize the revenue trends from the report.")
print(response)